# ReCaVSR int8 trên Kaggle (chạy tay, không cần app Windows)

Cùng một worker mà ứng dụng Windows đẩy lên Kaggle (`kaggle_jobs/recavsr_worker.py`):
clone ComfyUI + node pack [ComfyUI-ReCaVSR](https://github.com/PedroMarinhoDev/ComfyUI-ReCaVSR) → tải weight
(transformer **int8 ConvRot** 6,27 GB + `prompt` + `flashdecoder`) → chạy ComfyUI headless → upscale → lưu kết quả
vào `/kaggle/working`.

**Chuẩn bị:** *Accelerator = GPU T4 x2*, ***Internet = On*** (bắt buộc: tải node + weight), trống ≥ 8 GB.

⚠️ Điểm cần biết trước:
* Node pack **cần `triton`**: thiếu triton là node không import được (lỗi ở `flashdecoder.py`) — worker cài triton bắt buộc.
* Model **4x** rất tốn VRAM: 960×480 → 3840×1920 cần ~18–21 GB (không vừa T4 16 GB). Trên T4 nên dùng input nhỏ hơn
  hoặc `--scale 2`. `--lowvram` bật mặc định; thêm `--kv-offload` khi vẫn thiếu.
* int8 ConvRot cần GPU compute capability ≥ 7.5 → **T4 đúng ngưỡng** (chạy được).

In [ ]:
#@title 1) Kiểm tra môi trường
import os, shutil, subprocess, sys
from pathlib import Path

!nvidia-smi | head -12
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| GPU:", torch.cuda.device_count())
WORK = Path("/kaggle/working" if Path("/kaggle/working").is_dir() else ".")
print("working:", WORK, "| đĩa trống:", round(shutil.disk_usage(WORK).free / 1073741824, 1), "GB")

In [ ]:
#@title 2) Clone repo này (chứa worker + module)
REPO_URL = "https://github.com/hoannguyen171/UPSCALEVIDEO.git"  #@param {type:"string"}
CUI = WORK / "valo-upscale-gpu"

if not (CUI / ".git").exists():
    !git clone --depth 1 -q "{REPO_URL}" "{CUI}"
else:
    !git -C "{CUI}" pull -q --ff-only
%cd {CUI}
!git log --oneline -1
!pip install -q huggingface_hub

In [ ]:
#@title 3) Tải weight ReCaVSR (int8) + cài node pack (làm trước cho thấy tiến trình)
import sys
sys.path.insert(0, str(CUI))
from valo_upscale import recavsr

COMFY = WORK / "ComfyUI"
# 3a) ComfyUI master (cần >= 0.44)
if not (COMFY / "main.py").exists():
    !git clone --depth 1 -q https://github.com/comfyanonymous/ComfyUI.git "{COMFY}"
req = (COMFY / "requirements.txt").read_text(encoding="utf-8").splitlines()
skip = ("torch", "torchvision", "torchaudio")
(COMFY / "requirements-notorch.txt").write_text(
    "\n".join(l for l in req if l.split("==")[0].strip() not in skip), encoding="utf-8")
!pip install -q -r "{COMFY}/requirements-notorch.txt"
!pip install -q triton opencv-python-headless        # triton là BẮT BUỘC cho node ReCaVSR

# 3b) node pack
for cmd in recavsr.install_node_commands(str(COMFY)):
    print("$", " ".join(cmd[:4]), "…")
    subprocess.run(cmd, check=False)

# 3c) weight: int8 6,27 GB + prompt + flashdecoder → ComfyUI/models/recavsr
for path in recavsr.download_weights(COMFY / "models" / "recavsr", "int8",
                                    token=os.environ.get("HF_TOKEN"),
                                    source_dir="/kaggle/input" if Path("/kaggle/input").is_dir() else None,
                                    log=lambda t: print(t, flush=True)):
    print(f"  ✔ {Path(path).name} — {Path(path).stat().st_size / 1073741824:.2f} GB")

In [ ]:
#@title 4) Upscale (worker tự bật ComfyUI headless)
VIDEO = ""  #@param {type:"string"}
SCALE = 4  #@param [4, 2, 1.5] {type:"raw"}
PRECISION = "int8"  #@param ["int8", "bf16"]
COLOR_FIX = "none"  #@param ["none", "adain", "wavelet"]
KERNELS = "fast"  #@param ["fast", "exact"]
ENCODER = "auto"  #@param ["auto", "hevc nvenc", "h264 nvenc", "h264"]
KV_OFFLOAD = False  #@param {type:"boolean"}

import glob
video = VIDEO.strip()
if not video:
    found = sorted(glob.glob("/kaggle/input/**/*.mp4", recursive=True))
    video = found[0] if found else ""
print("video:", video or "CHƯA CÓ — điền VIDEO hoặc add dataset có .mp4 vào /kaggle/input")

if video:
    args = ["--video", video, "--scale", str(SCALE), "--precision", PRECISION,
            "--color-fix", COLOR_FIX, "--kernels", KERNELS, "--encoder", ENCODER, "--lowvram"]
    if KV_OFFLOAD:
        args.append("--kv-offload")
    !python kaggle_jobs/recavsr_worker.py {" ".join(args)}

In [ ]:
#@title 5) Xem kết quả
import json
from pathlib import Path
from IPython.display import Video, display
from valo_upscale import probe

summary_path = WORK / "valo_result.json"
summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
print(json.dumps(summary, ensure_ascii=False, indent=2))
if summary.get("output_path") and Path(summary["output_path"]).exists():
    path = Path(summary["output_path"])
    info = probe.get_media_info(path)
    print(f"{path.name}: {info['width']}x{info['height']} | {info['nb_frames']} frame | audio {info['has_audio']}")
    display(Video(str(path), embed=True, width=560))
    print("Nhớ bấm Save version để tải ở tab Output:", path)

## 6) Mẹo cho T4 16 GB

| Việc | Cách làm |
|---|---|
| Hết VRAM ở 4x | Giảm `SCALE` xuống 2 (hoặc 1.5), hoặc cắt clip ngắn hơn — VRAM chủ yếu do **kích thước đầu ra** quyết định |
| Vẫn thiếu VRAM | `KV_OFFLOAD = True` (giữ lịch sử attention ở RAM, kết quả y hệt) |
| Muốn lấy lại texture mịn hơn | (tuỳ chọn) `decoder = vae` — cần thêm `wan2.2_vae.safetensors` và chậm hơn 25–40× |
| Kernel `fast` không chạy | Đổi `KERNELS = "exact"` (chậm hơn, đúng tham chiếu) |
| Video ra chơi không được | `ENCODER = "h264"` (x264, tương thích mọi nơi; NVENC nhanh hơn nhưng HEVC có thể khó phát) |
| Không muốn tải lại 6,3 GB mỗi phiên | Up thư mục `models/recavsr` thành **Kaggle Dataset** rồi *Add Input*; worker tự symlink (`source_dir=/kaggle/input`) |
| Muốn chạy từ Windows mà không mở notebook | Dùng app: `python run_win.py` (đẩy chính worker này lên Kaggle, tự chia video qua tunnel và tải kết quả về) |

**Chưa kiểm chứng ở máy dev:** phần suy luận ReCaVSR (cần GPU). Đã kiểm: node pack nạp được trong ComfyUI 0.44,
workflow dựng bằng code trùng file API chính thức và VALID với node thật, weight đúng repo/đúng tên file,
lệnh chạy + hậu kiểm của app/worker, và cả **tunnel thật** (tải file qua URL công khai).